# TMDB Movie Data Analysis with Spark

Pipeline: fetch movie data from the TMDB API → clean it with PySpark → calculate KPIs → visualise.
Helper functions live in `src/`; this notebook runs them in order and shows the results.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import os
import sys
from pathlib import Path

# Make the project root importable, so "from src import ..." works from /notebooks
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Show log messages in the notebook output (stdout = normal, not red, output)
logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)   # hide Spark's internal connection messages
logger = logging.getLogger("notebook")

from src import config
from src.fetch import fetch_all_movies, save_raw_json

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis


## 1. Fetch movie data from the TMDB API
Raw responses (including cast and crew via `append_to_response=credits`) are saved to
`data/raw/movies.json`, so the API is only called when needed.

In [2]:
FETCH_FROM_API = True   # True = call TMDB now; False = reuse data/raw/movies.json

if FETCH_FROM_API or not config.RAW_MOVIES_FILE.exists():
    save_raw_json(fetch_all_movies())
else:
    logger.info("Using saved raw data: %s", config.RAW_MOVIES_FILE)

2026-11-04 12:27:38,412 WARNING src.fetch: Movie 0: not found (404) - skipped
2026-11-04 12:27:38,665 INFO src.fetch: Movie 299534: fetched 'Avengers: Endgame'
2026-11-04 12:27:38,889 INFO src.fetch: Movie 19995: fetched 'Avatar'
2026-11-04 12:27:39,061 INFO src.fetch: Movie 140607: fetched 'Star Wars: The Force Awakens'
2026-11-04 12:27:39,268 INFO src.fetch: Movie 299536: fetched 'Avengers: Infinity War'
2026-11-04 12:27:39,443 INFO src.fetch: Movie 597: fetched 'Titanic'
2026-11-04 12:27:39,623 INFO src.fetch: Movie 135397: fetched 'Jurassic World'
2026-11-04 12:27:39,793 INFO src.fetch: Movie 420818: fetched 'The Lion King'
2026-11-04 12:27:39,992 INFO src.fetch: Movie 24428: fetched 'The Avengers'
2026-11-04 12:27:40,159 INFO src.fetch: Movie 168259: fetched 'Furious 7'
2026-11-04 12:27:40,362 INFO src.fetch: Movie 99861: fetched 'Avengers: Age of Ultron'
2026-11-04 12:27:40,556 INFO src.fetch: Movie 284054: fetched 'Black Panther'
2026-11-04 12:27:40,727 INFO src.fetch: Movie 124

In [3]:
from pyspark.sql import SparkSession

# Spark's Python workers must use the same Python as this notebook (not the Store "python3")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("tmdb-analysis")
    .master("local[*]")                              # run on this PC, all CPU cores
    .config("spark.sql.shuffle.partitions", "4")     # default is 200: far too many for 18 rows
    .config("spark.sql.session.timeZone", "UTC")     # consistent dates on any PC
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")              # hide INFO/WARN noise in the notebook

print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark 4.3.0 | shuffle partitions: 4


In [4]:
raw_df = spark.read.json(config.RAW_MOVIES_FILE.as_posix(), multiLine=True)

print(f"Rows: {raw_df.count()} | Columns: {len(raw_df.columns)}")
raw_df.select("id", "title", "budget", "revenue").show(truncate=False)

Rows: 18 | Columns: 28
+------+--------------------------------------------+---------+----------+
|id    |title                                       |budget   |revenue   |
+------+--------------------------------------------+---------+----------+
|299534|Avengers: Endgame                           |356000000|2942186933|
|19995 |Avatar                                      |237000000|2923706026|
|140607|Star Wars: The Force Awakens                |245000000|2068223624|
|299536|Avengers: Infinity War                      |300000000|2052415039|
|597   |Titanic                                     |200000000|2264162353|
|135397|Jurassic World                              |150000000|1671537444|
|420818|The Lion King                               |260000000|1662020819|
|24428 |The Avengers                                |220000000|1518815515|
|168259|Furious 7                                   |190000000|1515400000|
|99861 |Avengers: Age of Ultron                     |235000000|1405403694|
|2

### 1.1 Inspect the raw data
Check the columns, data types and nested (JSON-like) fields before cleaning.

In [5]:
print(sorted(raw_df.columns))

['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'credits', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


In [6]:
raw_df.printSchema(level=2)   # level=2 hides the very long cast/crew details

root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |-- crew: array (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_com

In [7]:
nested_cols = [field.name for field in raw_df.schema.fields
               if field.dataType.typeName() in ("struct", "array")]
print("Nested columns:", nested_cols)

Nested columns: ['belongs_to_collection', 'credits', 'genres', 'origin_country', 'production_companies', 'production_countries', 'spoken_languages']


In [8]:
from pyspark.sql import functions as F

raw_df.select(
    "title",
    F.col("belongs_to_collection.name").alias("collection"),
    F.col("genres.name").alias("genre_names"),
    F.size("credits.cast").alias("cast_size"),
    F.size("credits.crew").alias("crew_size"),
).show(truncate=False)

+--------------------------------------------+-----------------------------------+-----------------------------------------------+---------+---------+
|title                                       |collection                         |genre_names                                    |cast_size|crew_size|
+--------------------------------------------+-----------------------------------+-----------------------------------------------+---------+---------+
|Avengers: Endgame                           |The Avengers Collection            |[Adventure, Science Fiction, Action]           |103      |614      |
|Avatar                                      |Avatar Collection                  |[Science Fiction, Action, Adventure]           |68       |999      |
|Star Wars: The Force Awakens                |Star Wars Collection               |[Adventure, Action, Science Fiction]           |99       |270      |
|Avengers: Infinity War                      |The Avengers Collection            |[Adventure, 

In [9]:
to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
json_like = ["belongs_to_collection", "genres", "production_countries",
             "production_companies", "spoken_languages"]

missing = [c for c in to_drop + json_like + ["credits"] if c not in raw_df.columns]
print("Missing from raw_df:", missing)

Missing from raw_df: []


## 2. Data cleaning and preprocessing
Cleaning functions live in `src/cleaning.py`. Each step returns a new DataFrame; `raw_df` is kept unchanged.

### 2.1 Drop irrelevant columns

In [10]:
from src import cleaning

df = cleaning.drop_irrelevant_columns(raw_df)

print(f"Columns: {len(raw_df.columns)} -> {len(df.columns)}")
print("Dropped:", sorted(set(raw_df.columns) - set(df.columns)))

Columns: 28 -> 23
Dropped: ['adult', 'homepage', 'imdb_id', 'original_title', 'video']


### 2.2 Flatten JSON-like columns
Keep the collection name, and join genre / language / country / company names with `"|"`.
Empty lists become NULL (not empty text) so missing data stays visible.

In [11]:
df = cleaning.flatten_json_columns(df)

flat_cols = ["belongs_to_collection", *config.JSON_NAME_FIELDS]
df.select("title", *flat_cols).show(truncate=False)
df.select(*flat_cols).printSchema()

+--------------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------------+---------------------------------------+------------------------------------------------------------------------------------+
|title                                       |belongs_to_collection              |genres                                   |spoken_languages                             |production_countries                   |production_companies                                                                |
+--------------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------------+---------------------------------------+------------------------------------------------------------------------------------+
|Avengers: Endgame                           |The Avengers Collection            |Adventure|Science Fiction|Acti

### 2.3 Extract cast and crew from `credits`
`cast` = all cast names joined with `"|"`; `director` = crew members whose job is exactly "Director";
`cast_size` / `crew_size` = number of people. The nested `credits` column is then dropped.

In [12]:
from pyspark.sql import functions as F

df = cleaning.extract_credits(df)

df.select(
    "title", "director", "cast_size", "crew_size",
    F.substring("cast", 1, 60).alias("cast (first 60 chars)"),
).show(truncate=False)

print("credits still in df?", "credits" in df.columns)
df.select("cast", "cast_size", "director", "crew_size").printSchema()

+--------------------------------------------+-----------------------+---------+---------+------------------------------------------------------------+
|title                                       |director               |cast_size|crew_size|cast (first 60 chars)                                       |
+--------------------------------------------+-----------------------+---------+---------+------------------------------------------------------------+
|Avengers: Endgame                           |Joe Russo|Anthony Russo|103      |614      |Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chris Hemsworth|S|
|Avatar                                      |James Cameron          |68       |999      |Sam Worthington|Zoe Saldaña|Sigourney Weaver|Stephen Lang|Mi|
|Star Wars: The Force Awakens                |J.J. Abrams            |99       |270      |Daisy Ridley|John Boyega|Oscar Isaac|Adam Driver|Carrie Fish|
|Avengers: Infinity War                      |Anthony Russo|Joe Russo|69       |740     

### 2.4 Inspect the extracted columns
Spark has no `value_counts()`, so `cleaning.value_counts()` groups and counts.
`"|"`-separated columns are split first so each genre / language / company is counted on its own.

In [13]:
inspect_cols = ["belongs_to_collection", "genres", "spoken_languages",
                "production_countries", "production_companies", "director"]

for column in inspect_cols:
    print(f"=== {column} ===")
    cleaning.value_counts(df, column, split=(column != "belongs_to_collection")).show(25, truncate=False)

=== belongs_to_collection ===
+-----------------------------------+-----+
|belongs_to_collection              |count|
+-----------------------------------+-----+
|The Avengers Collection            |4    |
|Frozen Collection                  |2    |
|Jurassic Park Collection           |2    |
|Star Wars Collection               |2    |
|NULL                               |2    |
|Avatar Collection                  |1    |
|Black Panther Collection           |1    |
|Harry Potter Collection            |1    |
|The Fast and the Furious Collection|1    |
|The Incredibles Collection         |1    |
|The Lion King (Reboot) Collection  |1    |
+-----------------------------------+-----+

=== genres ===
+---------------+-----+
|genres         |count|
+---------------+-----+
|Adventure      |15   |
|Action         |10   |
|Science Fiction|10   |
|Family         |5    |
|Animation      |4    |
|Fantasy        |4    |
|Thriller       |3    |
|Drama          |2    |
|Romance        |2    |
|Comed

In [14]:
cleaning.value_counts(df, "cast", split=True).show(10, truncate=False)

+-----------------+-----+
|cast             |count|
+-----------------+-----+
|Samuel L. Jackson|5    |
|Stan Lee         |5    |
|Andy Serkis      |4    |
|Chris Evans      |4    |
|Chris Hemsworth  |4    |
|Chris Pratt      |4    |
|Cobie Smulders   |4    |
|Fred Tatasciore  |4    |
|Mark Ruffalo     |4    |
|Robert Downey Jr.|4    |
+-----------------+-----+
only showing top 10 rows


**Findings:**
- No spelling variants or duplicate labels were found (e.g. only "Science Fiction", never "Sci-Fi"), so no relabelling is needed.
- 16 of 18 movies belong to a collection; only *Titanic* and *Beauty and the Beast* are standalone.
  The franchise vs standalone comparison therefore rests on just 2 standalone films.
- Every movie is in English and produced in the USA; Adventure (15/18) dominates the genres.
- Marvel Studios produced 5 of the 18 movies, the most of any company.
- Directors such as the Russo brothers and Chris Buck / Jennifer Lee co-direct, so `director` can hold two names.

### 2.5 Convert datatypes
Numbers are enforced with `try_cast` and `release_date` is parsed with `try_to_date`.
Spark 4 runs in strict (ANSI) mode, where a plain `cast` crashes on one bad value;
the `try_` versions turn invalid values into NULL instead (the "set invalid values to NaN" rule).

In [15]:
type_cols = [*config.COLUMN_TYPES, "release_date"]

def null_counts(frame):
    """How many NULLs each converted column has."""
    return frame.select([F.count_if(F.col(c).isNull()).alias(c) for c in type_cols]).first().asDict()

types_before, nulls_before = dict(df.dtypes), null_counts(df)
df = cleaning.convert_datatypes(df)
types_after, nulls_after = dict(df.dtypes), null_counts(df)

for c in type_cols:
    print(f"{c:<13} {types_before[c]:<7} -> {types_after[c]:<7} | NULLs {nulls_before[c]} -> {nulls_after[c]}")

id            bigint  -> bigint  | NULLs 0 -> 0
budget        bigint  -> double  | NULLs 0 -> 0
revenue       bigint  -> double  | NULLs 0 -> 0
popularity    double  -> double  | NULLs 0 -> 0
runtime       bigint  -> int     | NULLs 0 -> 0
vote_count    bigint  -> int     | NULLs 0 -> 0
vote_average  double  -> double  | NULLs 0 -> 0
release_date  string  -> date    | NULLs 0 -> 0


### 2.6 Replace unrealistic values and convert to million USD
TMDB uses 0 for "unknown", so budget / revenue / runtime values ≤ 0 become NULL
(not inferred: with 18 movies any estimate would be invented data).
Budget and revenue are then converted to million USD (`budget_musd`, `revenue_musd`), unrounded.

In [16]:
zero_counts = df.select(
    [F.count_if(F.col(c) <= 0).alias(c) for c in config.ZERO_MEANS_MISSING]
).first().asDict()
print("Values <= 0 before:", zero_counts)

df = cleaning.replace_unrealistic_values(df)
df = cleaning.convert_to_millions(df)

print("budget/revenue still in df?", [c for c in ("budget", "revenue") if c in df.columns])
df.select("title", "budget_musd", "revenue_musd", "runtime").show(5, truncate=False)

Values <= 0 before: {'budget': 0, 'revenue': 0, 'runtime': 0}
budget/revenue still in df? []
+----------------------------+-----------+------------+-------+
|title                       |budget_musd|revenue_musd|runtime|
+----------------------------+-----------+------------+-------+
|Avengers: Endgame           |356.0      |2942.186933 |181    |
|Avatar                      |237.0      |2923.706026 |162    |
|Star Wars: The Force Awakens|245.0      |2068.223624 |136    |
|Avengers: Infinity War      |300.0      |2052.415039 |149    |
|Titanic                     |200.0      |2264.162353 |194    |
+----------------------------+-----------+------------+-------+
only showing top 5 rows


### 2.7 Zero-vote ratings and placeholder text
`vote_average` is set to NULL when `vote_count` is 0 (a rating with no votes is meaningless).
Placeholder text in `overview` / `tagline` (e.g. "No Data", empty text) becomes NULL.

In [17]:
def quality_report(frame):
    """Counts of zero-vote movies, placeholder texts and NULLs, as evidence before/after cleaning."""
    return frame.select(
        F.count_if(F.col("vote_count") == 0).alias("zero_votes"),
        *[F.count_if(F.lower(F.trim(c)).isin(config.TEXT_PLACEHOLDERS)).alias(f"{c}_placeholders")
          for c in config.TEXT_COLUMNS],
        *[F.count_if(F.col(c).isNull()).alias(f"{c}_nulls")
          for c in ["vote_average", *config.TEXT_COLUMNS]],
    ).first().asDict()

print("Before:", quality_report(df))
df = cleaning.clear_unrated_vote_average(df)
df = cleaning.replace_text_placeholders(df)
print("After: ", quality_report(df))

Before: {'zero_votes': 0, 'overview_placeholders': 0, 'tagline_placeholders': 0, 'vote_average_nulls': 0, 'overview_nulls': 0, 'tagline_nulls': 0}
After:  {'zero_votes': 0, 'overview_placeholders': 0, 'tagline_placeholders': 0, 'vote_average_nulls': 0, 'overview_nulls': 0, 'tagline_nulls': 0}


### 2.8 Remove bad rows
Drop rows without `id`/`title`, then duplicate movies (same `id`), then rows with fewer than 10 non-null values.
The row count is logged after each rule.

In [18]:
df = cleaning.remove_bad_rows(df)

2026-11-04 12:28:04,639 INFO src.cleaning: Rows at start: 18
2026-11-04 12:28:04,873 INFO src.cleaning: After dropping rows without ['id', 'title']: 18
2026-11-04 12:28:05,192 INFO src.cleaning: After removing duplicate ids: 18
2026-11-04 12:28:06,155 INFO src.cleaning: After keeping rows with >= 10 non-null values: 18


**Finding:** all 18 fetched movies are unique, have an id and title, and have at least 10 non-null values,
so no rows were removed.

### 2.9 Keep only released movies
Check the status values, keep `"Released"` movies, then drop `status` (it no longer carries information).

In [19]:
cleaning.value_counts(df, "status").show()

df = cleaning.keep_released_movies(df)
print("status still in df?", "status" in df.columns)

+--------+-----+
|  status|count|
+--------+-----+
|Released|   18|
+--------+-----+

2026-11-04 12:31:39,925 INFO src.cleaning: Kept 18 of 18 movies with status 'Released'
status still in df? False


### 2.10 Final column order
Select the 22 required columns in the required order (columns not in the list are dropped).
"Reset index" has no Spark equivalent: Spark DataFrames have no row index, so row order is set
explicitly with `orderBy` wherever it matters (e.g. the KPI rankings).

In [20]:
df = cleaning.select_final_columns(df)

print("Columns:", len(df.columns), "| matches required order:", df.columns == config.FINAL_COLUMNS)
df.printSchema()
df.select("id", "title", "release_date", "budget_musd", "revenue_musd",
          "vote_average", "director").show(5, truncate=False)

2026-11-04 12:35:04,621 INFO src.cleaning: Not in the final layout (dropped): ['backdrop_path', 'origin_country', 'softcore']
Columns: 22 | matches required order: True
root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- genres: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- budget_musd: double (nullable = true)
 |-- revenue_musd: double (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- popularity: double (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- overview: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- cast

### 2.11 Save the cleaned data (checkpoint)
Saved as Parquet so Phases 3–4 can reload the clean data with its schema intact
(no API call or re-cleaning). `data/` is ignored by Git and recreated by this notebook.

In [21]:
cleaning.save_clean_data(df)

clean_df = cleaning.load_clean_data(spark)
print("Rows:", clean_df.count(), "| Columns:", len(clean_df.columns),
      "| Same schema as df:", clean_df.schema == df.schema)

2026-11-04 12:44:29,029 INFO src.cleaning: Saved cleaned data to C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis\data\processed\movies_clean.parquet
Rows: 18 | Columns: 22 | Same schema as df: True


### Phase 2 summary
| Step | Rule | Result on this dataset |
|---|---|---|
| 2.1 | Drop 5 irrelevant columns | 28 → 23 columns |
| 2.2 | Flatten JSON-like columns with `"\|"` | 5 nested columns → text; empty lists → NULL |
| 2.3 | Extract cast / director / sizes from `credits` | 4 new columns; co-directors kept (e.g. Russo brothers) |
| 2.4 | Inspect value counts | No spelling variants; 16 franchise vs 2 standalone films |
| 2.5 | Convert datatypes (`try_cast`, `try_to_date`) | Types enforced; no values lost (NULL counts unchanged) |
| 2.6 | 0 budget/revenue/runtime → NULL; convert to million USD | 0 unrealistic values found |
| 2.7 | Zero-vote ratings and placeholder text → NULL | None found |
| 2.8 | Missing id/title, duplicates, < 10 non-null values | 18 → 18 rows |
| 2.9 | Keep "Released" only | 18 of 18 kept |
| 2.10 | Final 22 columns in required order | Order verified; no nested types left |
| 2.11 | Save Parquet checkpoint | Reloaded with an identical schema |

Every rule stays in the pipeline even where it changed nothing, so future fetches are protected.

## 3. KPI implementation and analysis
Starts from the Phase 2 checkpoint (`data/processed/movies_clean.parquet`).
Profit and ROI are calculated once, in `src/kpis.py`, and reused by every ranking, comparison and chart.

In [23]:
from src import kpis

movies_df = kpis.add_financial_metrics(cleaning.load_clean_data(spark))

movies_df.select(
    "title", "budget_musd", "revenue_musd",
    F.round("profit_musd", 2).alias("profit_musd"),
    F.round("roi", 2).alias("roi"),
).show(5, truncate=False)

+--------------------------------------------+-----------+------------+-----------+-----+
|title                                       |budget_musd|revenue_musd|profit_musd|roi  |
+--------------------------------------------+-----------+------------+-----------+-----+
|Titanic                                     |200.0      |2264.162353 |2064.16    |11.32|
|Harry Potter and the Deathly Hallows: Part 2|125.0      |1341.511219 |1216.51    |10.73|
|Avatar                                      |237.0      |2923.706026 |2686.71    |12.34|
|The Avengers                                |220.0      |1518.815515 |1298.82    |6.9  |
|Avengers: Age of Ultron                     |235.0      |1405.403694 |1170.4     |5.98 |
+--------------------------------------------+-----------+------------+-----------+-----+
only showing top 5 rows


### 3.1 Ranking helper and UDF
`kpis.rank_movies()` ranks movies by any metric (skipping NULLs, with an optional extra filter) using a window
`rank()`, because ranking compares all rows. A Spark UDF (`kpis.format_metric`) formats each result row
for readability; it runs only on the top-N rows, so the Python overhead stays small.

In [24]:
kpis.rank_movies(movies_df, "revenue_musd").show(truncate=False)

+----+----------------------------+------------+-------+
|rank|title                       |revenue_musd|display|
+----+----------------------------+------------+-------+
|1   |Avengers: Endgame           |2942.186933 |$2.94B |
|2   |Avatar                      |2923.706026 |$2.92B |
|3   |Titanic                     |2264.162353 |$2.26B |
|4   |Star Wars: The Force Awakens|2068.223624 |$2.07B |
|5   |Avengers: Infinity War      |2052.415039 |$2.05B |
+----+----------------------------+------------+-------+

